# Solar filament instance segmentation — full pipeline (final submission)

This notebook rebuilds our final Kaggle submission from a fresh clone of the repo: code, data and **model weights
are all in the repo**, nothing else is downloaded except Python packages.

**Pipeline** (`src/final_pipeline.py`, exp101 — val official PQ **0.4480**, Dice 0.6520; public LB 0.39):
1. Two Mask R-CNN models at native 2048 x 2048 (no tiling): X101-FPN with a 56x56 mask head and R50-FPN, trained on
   every annotator set of the 566 training files (frozen split, 27-day time blocks).
2. Test-time augmentation: each model predicts the original image and its horizontal, vertical and double flip.
3. Per model the 4 views are fused (IoU >= 0.5, mask union, mean score); then the two models are fused
   (IoU >= 0.3, mask intersection, mean score).
4. Post-processing: closing (5 px), score >= 0.6, area >= 50 px², overlapping masks removed.
5. CSV in the competition format (`filament_id,segmentation_rle`), validated.

**Runtime:** Runtime -> Change runtime type -> **GPU** (T4 works; ~15 GB). Then Run all.
Timing on a T4 (estimate): install ~8 min, test set (180 images x 8 forward passes) ~30–45 min,
optional val scoring (141 images) ~25–35 min. Set `RUN_VAL = False` to skip it.

In [ ]:
RUN_VAL = True        # also score the frozen val split (official PQ + Dice), ~25-35 min on a T4
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo (code + data + weights)

In [ ]:
import os
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 2. Install dependencies (Detectron2 is built from source, ~5–8 min)

In [ ]:
!pip install -q -r requirements.txt
# same Detectron2 commit as requirements-lock.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git@1e3e13bbf607b54f62205c4c33922521822fb298'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 3. Join the weight parts (size + SHA-256 checked) and run the quick tests

In [ ]:
!python src/weights_io.py unpack --dir weights
!python -m pytest -q -p no:warnings tests/test_final_pipeline.py tests/test_flip.py tests/test_rle_and_submission.py tests/test_official_metric.py

## 4. Predict the 180 test images -> submission CSV
Prints per-image latency (median / p95 after 2 warm-up images, including image loading, 8 forward passes, fusion
and post-processing), peak GPU memory and the validator result.

In [ ]:
!python src/final_pipeline.py predict --split test --out submissions/submission_final.csv --raw-out output/raw_preds_final_test.json.gz --report output/report_test.json

## 5. Reproduction check
The fused raw predictions of the submitted run (exp101) are stored in the repo. Rebuild that CSV on the CPU and
compare it with the one just made (masks matched per image at IoU > 0.5). Small differences can come from a
different GPU / library version; "identical" is expected only on the same setup.

In [ ]:
!python src/submit_from_raw.py "Result Phase 6/Mirul/raw_preds_tta2_m28_r50_test.json.gz" submissions/submission_exp101_from_raw.csv --score-thr 0.6
!python src/final_pipeline.py compare --a submissions/submission_final.csv --b submissions/submission_exp101_from_raw.csv

## 6. (Optional) Official PQ + Dice on the frozen val split (expected ~0.448 / ~0.652)

In [ ]:
if RUN_VAL:
    !python src/final_pipeline.py predict --split val --report output/report_val.json

## 7. A look at the predictions (val image: annotator 1 vs our masks)

In [ ]:
import sys, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, "src")
from final_pipeline import load_predictors, predict
from annotations import load_gt_rles
from preprocess import image_path
from split import load_split
from pycocotools import mask as coco_mask

stems = load_split()[1][:2]
preds = predict([image_path(s, "train") for s in stems], load_predictors(Path("weights")))[1]
gt = load_gt_rles(stems)

def overlay(img, rles, color):
    out = img.copy()
    for r in rles:
        m = coco_mask.decode(r).astype(bool)
        out[m] = (0.5 * out[m] + 0.5 * np.array(color)).astype(np.uint8)
    return out

fig, ax = plt.subplots(len(stems), 2, figsize=(12, 6 * len(stems)))
for i, s in enumerate(stems):
    img = cv2.cvtColor(cv2.imread(str(image_path(s, "train"))), cv2.COLOR_BGR2RGB)
    ax[i, 0].imshow(overlay(img, gt.get(s, []), (0, 255, 0))); ax[i, 0].set_title(f"{s}: annotator 1 (green)")
    ax[i, 1].imshow(overlay(img, preds.get(s, []), (255, 0, 0))); ax[i, 1].set_title(f"{s}: our masks (red)")
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.show()